In [3]:
#Spatial Analysis - German Potential Industrial Symbiosis Partners 

import pandas as pd
import numpy as np

from sklearn.neighbors import BallTree

import folium
from folium import plugins
from folium.plugins import HeatMap

import warnings
warnings.filterwarnings("default")


# 1) Load and clean CSV 

INPUT_CSV = "Abfall2024.csv"
YEAR_FILTER = 2024

CLEANED_OUT = "Abfall2024_cleaned.csv"
COLLAB_OUT  = "collaborations_2024.csv"

MAP1_OUT = "Germany_Collaborations_ByDistance.html"
MAP2_OUT = "Germany_Collaboration_Density_ByDistance.html"

# Column names in CSV file
COL_YEAR   = "Berichtsjahr"
COL_STATE  = "Bundesland"                      
COL_NAME   = "Name_Betrieb"
COL_ZIP    = "Adresse_PLZ"
COL_CITY   = "Adresse_Ort"
COL_LAT    = "Koordinaten_geo_lat_wgs84"
COL_LON    = "Koordinaten_geo_long_wgs84"
COL_SECTOR = "Branche"

df_raw = pd.read_csv(
    INPUT_CSV,
    sep=";",
    encoding="utf-8-sig",     
    low_memory=False
)

needed_cols = [COL_YEAR, COL_STATE, COL_NAME, COL_ZIP, COL_CITY, COL_LAT, COL_LON, COL_SECTOR]
missing = [c for c in needed_cols if c not in df_raw.columns]
if missing:
    raise ValueError(f"Missing columns in CSV: {missing}")

df = df_raw[needed_cols].copy()
df = df[df[COL_YEAR] == YEAR_FILTER].copy()

for c in [COL_STATE, COL_NAME, COL_ZIP, COL_CITY, COL_SECTOR]:
    df[c] = df[c].astype("string").str.strip()

df[COL_LAT] = pd.to_numeric(df[COL_LAT], errors="coerce")
df[COL_LON] = pd.to_numeric(df[COL_LON], errors="coerce")

df = df.dropna(subset=[COL_LAT, COL_LON, COL_STATE, COL_NAME, COL_SECTOR]).copy()
df = df[df[COL_LAT].between(-90, 90) & df[COL_LON].between(-180, 180)].copy()

# Aggregation to reduce duplicates
df = (
    df.groupby([COL_STATE, COL_ZIP, COL_CITY, COL_NAME, COL_SECTOR], as_index=False, dropna=False)
      .agg({COL_LAT: "mean", COL_LON: "mean", COL_YEAR: "first"})
)

df.to_csv(CLEANED_OUT, index=False, sep=";")
print(f"Cleaned rows for {YEAR_FILTER}: {len(df)}\nSaved to: {CLEANED_OUT}")


# 2) Collaboration analysis

# Distance thresholds (in km)
SHORT_MAX_KM = 15.4 
LONG_MAX_KM = 32.6

# Use the largest threshold as the search radius
SEARCH_RADIUS_KM = LONG_MAX_KM

EARTH_RADIUS_KM = 6371.0

def classify_distance_km(d_km):
    if d_km <= SHORT_MAX_KM:
        return "short"
    if d_km <= LONG_MAX_KM:
        return "long"
    return None


# Sector compatibility rules 
allowed_partners_by_sector = {
    "Abfall- und Abwasserbewirtschaftung": {
        "Chemische Industrie", "Energiesektor", "Intensivtierhaltung und Aquakultur",
        "Lebensmittelindustrie", "Metallindustrie", "Mineralverarbeitende Industrie",
        "Papier- und Holzindustrie", "Sonstige Industriezweige"
    },
    "Chemische Industrie": {
        "Abfall- und Abwasserbewirtschaftung", "Energiesektor", "Lebensmittelindustrie",
        "Metallindustrie", "Mineralverarbeitende Industrie", "Papier- und Holzindustrie",
        "Sonstige Industriezweige", "Sonstige Industriezweig"
    },
    "Energiesektor": {
        "Abfall- und Abwasserbewirtschaftung", "Chemische Industrie", "Energiesektor",
        "Intensivtierhaltung und Aquakultur", "Lebensmittelindustrie", "Metallindustrie",
        "Mineralverarbeitende Industrie", "Papier- und Holzindustrie", "Sonstige Industriezweige"
    },
    "Intensivtierhaltung und Aquakultur": {
        "Abfall- und Abwasserbewirtschaftung", "Energiesektor", "Lebensmittelindustrie",
        "Sonstige Industriezweige"
    },
    "Lebensmittelindustrie": {
        "Abfall- und Abwasserbewirtschaftung", "Chemische Industrie", "Energiesektor",
        "Intensivtierhaltung und Aquakultur", "Metallindustrie", "Mineralverarbeitende Industrie",
        "Papier- und Holzindustrie", "Sonstige Industriezweige"
    },
    "Metallindustrie": {
        "Abfall- und Abwasserbewirtschaftung", "Chemische Industrie", "Energiesektor",
        "Lebensmittelindustrie", "Mineralverarbeitende Industrie", "Papier- und Holzindustrie",
        "Sonstige Industriezweige"
    },
    "Mineralverarbeitende Industrie": {
        "Abfall- und Abwasserbewirtschaftung", "Chemische Industrie", "Energiesektor",
        "Lebensmittelindustrie", "Metallindustrie", "Papier- und Holzindustrie",
        "Sonstige Industriezweige"
    },
    "Papier- und Holzindustrie": {
        "Abfall- und Abwasserbewirtschaftung", "Chemische Industrie", "Energiesektor",
        "Lebensmittelindustrie", "Metallindustrie", "Mineralverarbeitende Industrie",
        "Sonstige Industriezweige"
    },
    "Sonstige Industriezweige": {
        "Abfall- und Abwasserbewirtschaftung", "Chemische Industrie", "Energiesektor",
        "Intensivtierhaltung und Aquakultur", "Lebensmittelindustrie", "Metallindustrie",
        "Mineralverarbeitende Industrie", "Papier- und Holzindustrie", "Sonstige Industriezweige"
    },
}

def sector_allowed_symmetric(sector_a, sector_b):
    a = (sector_a or "").strip()
    b = (sector_b or "").strip()
    return (b in allowed_partners_by_sector.get(a, set())) or (a in allowed_partners_by_sector.get(b, set()))


# State neighbor graph (with abbreviations) for cross-border collaborations
neighbors = {
    "BW": {"BY", "HE", "RP"},
    "BY": {"BW", "HE", "TH", "SN"},
    "BE": {"BB"},
    "BB": {"BE", "MV", "SN", "ST", "NI"},
    "HB": {"NI"},
    "HH": {"SH", "NI"},
    "HE": {"BW", "BY", "NI", "NW", "RP", "TH"},
    "MV": {"BB", "SH", "NI"},
    "NI": {"HB", "HH", "HE", "MV", "NW", "ST", "SH", "BB"},
    "NW": {"NI", "HE", "RP"},
    "RP": {"SL", "NW", "HE", "BW"},
    "SL": {"RP"},
    "SN": {"BY", "TH", "ST", "BB"},
    "ST": {"NI", "TH", "SN", "BB"},
    "SH": {"HH", "NI", "MV"},
    "TH": {"HE", "BY", "SN", "ST"},
}

def haversine_function(lat1, lon1, lat2, lon2):
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
    c = 2*np.arcsin(np.sqrt(a))
    return EARTH_RADIUS_KM * c


def build_collaborations(df_sites):

    # Rename to English internal columns
    d = df_sites.rename(columns={
        COL_YEAR: "year",
        COL_STATE: "state",
        COL_NAME: "company_name",
        COL_ZIP: "zip_code",
        COL_CITY: "city",
        COL_LAT: "latitude",
        COL_LON: "longitude",
        COL_SECTOR: "sector",
    }).copy()

    # Normalize text
    for c in ["state", "company_name", "zip_code", "city", "sector"]:
        d[c] = d[c].astype("string").str.strip()

    # Coordinates in radians
    coords = np.c_[np.radians(d["latitude"].to_numpy(float)),
                   np.radians(d["longitude"].to_numpy(float))]

    # Search radius in radians 
    radius_rad = SEARCH_RADIUS_KM / EARTH_RADIUS_KM

    results = []

    # A) Within-state collaborations
    for state, df_state in d.groupby("state", sort=True):
        if len(df_state) < 2:
            continue

        idx = df_state.index.to_numpy()
        coords_s = coords[idx]

        tree = BallTree(coords_s, metric="haversine")
        neigh_lists = tree.query_radius(coords_s, r=radius_rad)

        for i, js in enumerate(neigh_lists):
            js = js[js > i]  
            if js.size == 0:
                continue

            row_i = df_state.iloc[i]

            dists_km = haversine_function(
                coords_s[i, 0], coords_s[i, 1],
                coords_s[js, 0], coords_s[js, 1]
            )

            for jj, dist_km in zip(js, dists_km):
                transport = classify_distance_km(float(dist_km))
                if transport is None:
                    continue

                row_j = df_state.iloc[int(jj)]
                if not sector_allowed_symmetric(row_i["sector"], row_j["sector"]):
                    continue

                results.append({
                    "interaction_type": "within_state",
                    "transport_class": transport,
                    "distance_km": float(dist_km),

                    "state1": row_i["state"], "company1": row_i["company_name"],
                    "zip1": row_i["zip_code"], "city1": row_i["city"],
                    "sector1": row_i["sector"], "lat1": row_i["latitude"], "lon1": row_i["longitude"],

                     "state2": row_j["state"], "company2": row_j["company_name"],
                    "zip2": row_j["zip_code"], "city2": row_j["city"],
                    "sector2": row_j["sector"], "lat2": row_j["latitude"], "lon2": row_j["longitude"],

                    "center_lat": (row_i["latitude"] + row_j["latitude"]) / 2.0,
                    "center_lon": (row_i["longitude"] + row_j["longitude"]) / 2.0,
                })

    # B) Cross-state (neighboring states only)
    present_states = sorted(d["state"].dropna().unique().tolist())
    present_set = set(present_states)

    seen_state_pairs = set()

    for s1 in present_states:
        if s1 not in neighbors:
            continue
        for s2 in neighbors[s1]:
            if s2 not in present_set:
                continue

            pair = tuple(sorted([s1, s2]))
            if pair in seen_state_pairs:
                continue
            seen_state_pairs.add(pair)

            A = d[d["state"] == pair[0]].reset_index(drop=True)
            B = d[d["state"] == pair[1]].reset_index(drop=True)
            if len(A) == 0 or len(B) == 0:
                continue

            coordsA = np.c_[np.radians(A["latitude"].to_numpy(float)),
                            np.radians(A["longitude"].to_numpy(float))]
            coordsB = np.c_[np.radians(B["latitude"].to_numpy(float)),
                            np.radians(B["longitude"].to_numpy(float))]

            treeB = BallTree(coordsB, metric="haversine")
            neigh_lists = treeB.query_radius(coordsA, r=radius_rad)

            for iA, jsB in enumerate(neigh_lists):
                if len(jsB) == 0:
                    continue

                row_i = A.iloc[iA]

                dist_km = haversine_function(
                    coordsA[iA, 0], coordsA[iA, 1],
                    coordsB[jsB, 0], coordsB[jsB, 1]
                )

                for jB, dkm in zip(jsB, dist_km):
                    transport = classify_distance_km(float(dkm))
                    if transport is None:
                        continue

                    row_j = B.iloc[int(jB)]
                    if not sector_allowed_symmetric(row_i["sector"], row_j["sector"]):
                        continue

                    results.append({
                        "interaction_type": "cross_state_border",
                        "transport_class": transport,
                        "distance_km": float(dkm),

                        "state1": row_i["state"], "company1": row_i["company_name"],
                        "zip1": row_i["zip_code"], "city1": row_i["city"],
                        "sector1": row_i["sector"], "lat1": row_i["latitude"], "lon1": row_i["longitude"],

                        "state2": row_j["state"], "company2": row_j["company_name"],
                        "zip2": row_j["zip_code"], "city2": row_j["city"],
                        "sector2": row_j["sector"], "lat2": row_j["latitude"], "lon2": row_j["longitude"],

                        "center_lat": (row_i["latitude"] + row_j["latitude"]) / 2.0,
                        "center_lon": (row_i["longitude"] + row_j["longitude"]) / 2.0,
                    })

    df_collab = pd.DataFrame(results)

    # Final dedupe: treat (A,B)==(B,A) within same class/type
    if len(df_collab) > 0:
        def pair_key(r):
            a = (r["state1"], r["zip1"], r["company1"])
            b = (r["state2"], r["zip2"], r["company2"])
            return tuple(sorted([a, b])) + (r["transport_class"],)

        df_collab["pair_key"] = df_collab.apply(pair_key, axis=1)
        df_collab = df_collab.drop_duplicates("pair_key").drop(columns=["pair_key"]).reset_index(drop=True)

    return df_collab


df_collaborators = build_collaborations(df)
df_collaborators.to_csv(COLLAB_OUT, index=False, sep=";")

print("Collaborations found:", len(df_collaborators))
if len(df_collaborators) > 0:
    print(df_collaborators["transport_class"].value_counts())
    print(df_collaborators.groupby(["interaction_type", "transport_class"]).size())


# 3) MAP 1: Line map 

m1 = folium.Map(location=[51.1657, 10.4515], zoom_start=6, tiles="OpenStreetMap")

# Feature groups by distance thresholds
layer_short = folium.FeatureGroup(name="Short (<= 15.4 km)", show=True)
layer_long = folium.FeatureGroup(name="Long (15.4–32.6 km)", show=False)

# Midpoint marker clusters per layer
cluster_short = plugins.MarkerCluster(name="Midpoints: Short", show=False)
cluster_long = plugins.MarkerCluster(name="Midpoints: Long", show=False)

def add_line(layer, row, color):
    folium.PolyLine(
        locations=[(row["lat1"], row["lon1"]), (row["lat2"], row["lon2"])],
        weight=2,
        opacity=0.2,
        color=color
    ).add_to(layer)

def add_midpoint(cluster, row, color):
    popup = folium.Popup(
        f"<b>{row['transport_class']}</b> ({row['distance_km']:.2f} km)<br>"
        f"<b>A:</b> {row['company1']} ({row['state1']}, {row['zip1']} {row['city1']})<br>"
        f"<b>Sector:</b> {row['sector1']}<br><br>"
        f"<b>B:</b> {row['company2']} ({row['state2']}, {row['zip2']} {row['city2']})<br>"
        f"<b>Sector:</b> {row['sector2']}<br>"
        f"<b>Interaction:</b> {row['interaction_type']}",
        max_width=420
    )
    folium.CircleMarker(
        location=(row["center_lat"], row["center_lon"]),
        radius=4,
        color=color,
        fill=True,
        fill_opacity=0.7,
        popup=popup
    ).add_to(cluster)

if len(df_collaborators) > 0:
    for _, r in df_collaborators.iterrows():
        tc = r["transport_class"]

        if tc == "short":
            add_line(layer_short, r, "green")
            add_midpoint(cluster_short, r, "green")

        elif tc == "long":
            add_line(layer_long, r, "blue")
            add_midpoint(cluster_long, r, "blue")


layer_short.add_to(m1)
layer_long.add_to(m1)


cluster_short.add_to(m1)
cluster_long.add_to(m1)

folium.LayerControl(collapsed=False).add_to(m1)
m1.save(MAP1_OUT)
print(f"Line-based Map saved to: {MAP1_OUT}")

# 4) MAP 2: Density heatmap 

if len(df_collaborators) == 0:
    print("No collaborations found -> Heatmap not created.")
else:
    m2 = folium.Map(location=[51.1657, 10.4515], zoom_start=6, tiles="OpenStreetMap")

    # Split by distance thresholds
    df_short = df_collaborators[df_collaborators["transport_class"] == "short"].copy()
    df_long = df_collaborators[df_collaborators["transport_class"] == "long"].copy()

    def make_heat_data(df_sub, weight_mode="inverse_dist"):
        
        if len(df_sub) == 0:
            return []

        lat = df_sub["center_lat"].to_numpy(float)
        lon = df_sub["center_lon"].to_numpy(float)

        
        d = df_sub["distance_km"].to_numpy(float)
        w = 1.0 / (d + 0.2) 
        
        if (w.max() - w.min()) > 1e-9:
           w = (w - w.min()) / (w.max() - w.min())
        return [[float(la), float(lo), float(we)] for la, lo, we in zip(lat, lon, w)]

    WEIGHT_MODE = "inverse_dist"  

    heat_short  = make_heat_data(df_short,  weight_mode=WEIGHT_MODE)
    heat_long   = make_heat_data(df_long,   weight_mode=WEIGHT_MODE)
    heat_total = make_heat_data(df_collaborators, weight_mode=WEIGHT_MODE)

    fg_short  = folium.FeatureGroup(name="Density: Short Distance (<= 15.4 km)", show=True)
    fg_long   = folium.FeatureGroup(name="Density: Long Distance (15.2-32.6 km)", show=False)
    fg_total  = folium.FeatureGroup(name="Density: All Collaborations", show=False)

    heat_settings = dict(radius=12, blur=10, min_opacity=0.55, max_zoom=8)

    if heat_short:
        HeatMap(heat_short, **heat_settings).add_to(fg_short)
    if heat_long:
        HeatMap(heat_long, **heat_settings).add_to(fg_long)
    if heat_total:
        HeatMap(heat_total, **heat_settings).add_to(fg_total)

    fg_short.add_to(m2)
    fg_long.add_to(m2)
    fg_total.add_to(m2)

    folium.LayerControl(collapsed=False).add_to(m2)
    m2.save(MAP2_OUT)
    print(f"Heatmap saved to: {MAP2_OUT}")

# 5) Summary 

print("\n Summary: Collaborations by transport class")
if len(df_collaborators) > 0:
    display(df_collaborators["transport_class"].value_counts())

print("\n Summary: Collaborations by transport class")
if len(df_collaborators) > 0:
    display(df_collaborators["transport_class"].value_counts())

print("\n Summary: Top states by number of collaborations (short/long/total + % distribution)")
if len(df_collaborators) > 0:
    counts = (
        df_collaborators
        .pivot_table(
            index="state1",
            columns="transport_class",
            values="distance_km",
            aggfunc="count",
            fill_value=0
        )
        .rename(columns={"short": "collab_short", "long": "collab_long"})
    )

    for col in ["collab_short", "collab_long"]:
        if col not in counts.columns:
            counts[col] = 0

    counts["collab_total"] = counts["collab_short"] + counts["collab_long"]

    counts["pct_short"] = (counts["collab_short"] / counts["collab_total"] * 100).round(2)
    counts["pct_long"]  = (counts["collab_long"]  / counts["collab_total"] * 100).round(2)

    total_all = counts["collab_total"].sum()
    counts["pct_total_state"] = (counts["collab_total"] / total_all * 100).round(2)

    mean_total = df_collaborators.groupby("state1")["distance_km"].mean().rename("mean_distance_km"). round(2)

    summary_states = (
        counts.join(mean_total)
        .sort_values(["collab_total", "collab_long"], ascending=False)
    )

    display(summary_states.head(16))

Cleaned rows for 2024: 4593
Saved to: Abfall2024_cleaned.csv
Collaborations found: 128974
transport_class
long     91253
short    37721
Name: count, dtype: int64
interaction_type    transport_class
cross_state_border  long               13419
                    short               2520
within_state        long               77834
                    short              35201
dtype: int64
Line-based Map saved to: Germany_Collaborations_ByDistance.html
Heatmap saved to: Germany_Collaboration_Density_ByDistance.html

 Summary: Collaborations by transport class


transport_class
long     91253
short    37721
Name: count, dtype: int64


 Summary: Collaborations by transport class


transport_class
long     91253
short    37721
Name: count, dtype: int64


 Summary: Top states by number of collaborations (short/long/total + % distribution)


,collab_long,collab_short,collab_total,pct_short,pct_long,pct_total_state,mean_distance_km
state1,,,,,,,
NW,45241,15683,60924,25.74,74.26,47.24,20.91
BY,7946,3898,11844,32.91,67.09,9.18,19.29
BW,8377,3404,11781,28.89,71.11,9.13,20.02
HE,6398,3066,9464,32.40,67.60,7.34,19.36
NI,5561,1935,7496,25.81,74.19,5.81,20.94
SN,5172,1560,6732,23.17,76.83,5.22,21.77
ST,3385,2361,5746,41.09,58.91,4.46,18.32
RP,2308,1186,3494,33.94,66.06,2.71,18.99
HH,1340,1379,2719,50.72,49.28,2.11,16.21
